In [1]:
import pandas as pd
import os
import numpy as np
import re
from datetime import datetime, timedelta

In [2]:
pd.set_option('display.max_columns', None)

## **Step 0. READ Files**

#### <span style="color:#FF6347;">**READ**</span> mo_cancer file

In [67]:
file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", "Cancer", 'mo_cancer' + ".xlsx")
df = pd.read_excel(file_path)

In [68]:
df.head(3)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index
0,4333041185,50,454994176,2021-11-03,SCREEN,L,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaN,NaN,INDEX-1
1,4333041185,50,454994176,2021-11-03,SCREEN,R,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaN,NaN,INDEX-1
2,4333041185,51,451483856,2022-08-05,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,353483.0,2022-09-02,Malignant,INDEX


#### <span style="color:#FF6347;">**READ**</span> EHR file - procedure_notes, pathology_findings (Ki-67, ER, PR, HER2)

In [69]:
file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", "Cancer", "Cleaned", "pathology_findings" + ".xlsx")
pathology = pd.read_excel(file_path)
file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", "Cancer", "Cleaned", "procedure_notes" + ".xlsx")
procedure_notes = pd.read_excel(file_path)

In [70]:
pathology['PATHOLOGY_DATE'] = pd.to_datetime(pathology['PATHOLOGY_DATE'])
pathology = pathology.drop('LESION_CLASS', axis=1)
procedure_notes['ORDER_DATE'] = pd.to_datetime(procedure_notes['ORDER_DATE'])

In [71]:
pathology.columns

Index(['PATIENT_STUDY_ID', 'BX_ID', 'PATHOLOGY_DATE', 'PATHOLOGY_CD',
       'HISTROLOGY_GRADE', 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR',
       'HER2NEU', 'STAGE_T', 'STAGE_N', 'STAGE_M', 'STAGE_NUM',
       'MARGIN_STATUS'],
      dtype='object')

In [72]:
procedure_notes.columns

Index(['PATIENT_STUDY_ID', 'ORDER_DATE', 'NOTE_TEXT', 'Ki-67_Percent',
       'Ki-67_Grade'],
      dtype='object')

In [73]:
pathology.shape, procedure_notes.shape

((18349, 13), (3302, 5))

In [74]:
pathology[pathology['PATIENT_STUDY_ID']==4333041185]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,MARGIN_STATUS
888,4333041185,460259,2022-02-01,ID,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN
889,4333041185,460259,2022-02-01,II,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN
890,4333041185,354429,2022-08-30,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
891,4333041185,353483,2022-09-02,ID,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN
892,4333041185,353483,2022-09-02,II,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN
893,4333041185,359950,2022-10-24,II,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN


# **Step 1. Merge MO cancer with subtype information** 
### output: <span style="color:DodgerBlue;">**mo_cancer_subtype**</span>

### **1. Merge pathology_findings (ER, PR, HER2) and procedure_notes (Ki-67, NOTE_TEXT)** 

In [75]:
# 1. Assign a Unique ID to every row in pathology to guarantee row count
# We use this to map back to the original table at the end
pathology = pathology.reset_index().rename(columns={'index': 'pathology_row_id'})

# 2. Left Merge to create all potential matches
merged = pd.merge(
    pathology[['pathology_row_id', 'PATIENT_STUDY_ID', 'PATHOLOGY_DATE']], 
    procedure_notes, 
    on='PATIENT_STUDY_ID', 
    how='left'
)

# 4. Filter for your specific timeline constraint: ORDER_DATE >= PATHOLOGY_DATE
# We keep rows that are valid matches OR rows that have no notes at all (NaN)
valid_matches = merged[
    (merged['ORDER_DATE'] >= merged['PATHOLOGY_DATE']) | (merged['ORDER_DATE'].isna())
].copy()

# 5. Calculate the gap and find the CLOSEST subsequent note
valid_matches['days_diff'] = (valid_matches['ORDER_DATE'] - valid_matches['PATHOLOGY_DATE']).dt.days
valid_matches = valid_matches.sort_values(by=['pathology_row_id', 'days_diff'])

# 6. Keep only the single best note for each pathology row
best_matches = valid_matches.drop_duplicates(subset=['pathology_row_id'], keep='first')

# 7. Merge the Ki-67 data back to the original pathology dataframe
# This guarantees the final row count matches the original 'pathology' df
subtype_info = pd.merge(
    pathology, 
    best_matches[['pathology_row_id', 'ORDER_DATE', 'Ki-67_Percent', 'Ki-67_Grade', 'NOTE_TEXT']], 
    on='pathology_row_id', 
    how='left'
).drop(columns=['pathology_row_id'])

# 8. Validation
print(f"Original Pathology Rows: {len(pathology)}")
print(f"Final Linked Rows: {len(subtype_info)}")

Original Pathology Rows: 18349
Final Linked Rows: 18349


In [76]:
subtype_info.columns

Index(['PATIENT_STUDY_ID', 'BX_ID', 'PATHOLOGY_DATE', 'PATHOLOGY_CD',
       'HISTROLOGY_GRADE', 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR',
       'HER2NEU', 'STAGE_T', 'STAGE_N', 'STAGE_M', 'STAGE_NUM',
       'MARGIN_STATUS', 'ORDER_DATE', 'Ki-67_Percent', 'Ki-67_Grade',
       'NOTE_TEXT'],
      dtype='object')

In [77]:
subtype_info[subtype_info['PATIENT_STUDY_ID']==4333041185]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,MARGIN_STATUS,ORDER_DATE,Ki-67_Percent,Ki-67_Grade,NOTE_TEXT
888,4333041185,460259,2022-02-01,ID,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN,NaT,NaN,NaN,NaN
889,4333041185,460259,2022-02-01,II,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN,NaT,NaN,NaN,NaN
890,4333041185,354429,2022-08-30,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
891,4333041185,353483,2022-09-02,ID,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN,NaT,NaN,NaN,NaN
892,4333041185,353483,2022-09-02,II,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN,NaT,NaN,NaN,NaN
893,4333041185,359950,2022-10-24,II,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN,NaT,NaN,NaN,NaN


### **2. Merge MO cancer with pathology_findings, procedure_notes** (Ki-67, ER, PR, HER2)

#### <span style="color:blue;">**Main**</span>

In [78]:
df.columns

Index(['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate',
       'Study', 'SIDE', 'Series', 'COMPOSITION_NAME', 'FINDING_LOCATION',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE', 'BX_ID',
       'PATHOLOGY_DATE', 'LESION_CLASS', 'Index'],
      dtype='object')

In [79]:
subtype_info["PATHOLOGY_DATE"] = subtype_info["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

In [80]:
merge = pd.merge(
    df, 
    subtype_info,
    on=['PATIENT_STUDY_ID', 'BX_ID','PATHOLOGY_DATE'], 
    how='left'
)

In [81]:
idx_index_1 = merge["Index"]=="INDEX-1"
idx_index = merge["Index"]=="INDEX"
idx_dbt = merge["Series"]=="DBT"
idx_screen = merge["Study"]=="SCREEN"
idx_birads_0 = merge["FINDING_CATEGORY"]=="0 - Need additional imaging evaluation"
idx_birads_1 = merge["FINDING_CATEGORY"]=="1 - Negative"
idx_birads_2 = merge["FINDING_CATEGORY"]=="2 - Benign finding"

In [82]:
PIDs = merge['PATIENT_STUDY_ID'].unique()
len(PIDs)

7

In [83]:
print(merge['PROGESTERONE_RECEPTOR'].unique(), merge['ESTROGEN_RECEPTOR'].unique(), merge['HER2NEU'].unique(), merge['HISTROLOGY_GRADE'].unique())

[nan 'P'] [nan 'P'] [nan 'N'] [nan 'G2' 'GX']


In [84]:
merge[merge['PATIENT_STUDY_ID']==PIDs[0]].head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,MARGIN_STATUS,ORDER_DATE,Ki-67_Percent,Ki-67_Grade,NOTE_TEXT
0,4333041185,50,454994176,2021-11-03,SCREEN,L,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaN,NaN,INDEX-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
1,4333041185,50,454994176,2021-11-03,SCREEN,R,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaN,NaN,INDEX-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
2,4333041185,51,451483856,2022-08-05,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,353483.0,2022-09-02,Malignant,INDEX,ID,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN,NaT,NaN,NaN,NaN
3,4333041185,51,451483856,2022-08-05,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,353483.0,2022-09-02,Malignant,INDEX,II,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 2A,NaN,NaT,NaN,NaN,NaN
4,4333041185,51,451483856,2022-08-05,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,354429.0,2022-08-30,Benign,INDEX,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN


In [85]:
merge.columns

Index(['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate',
       'Study', 'SIDE', 'Series', 'COMPOSITION_NAME', 'FINDING_LOCATION',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE', 'BX_ID',
       'PATHOLOGY_DATE', 'LESION_CLASS', 'Index', 'PATHOLOGY_CD',
       'HISTROLOGY_GRADE', 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR',
       'HER2NEU', 'STAGE_T', 'STAGE_N', 'STAGE_M', 'STAGE_NUM',
       'MARGIN_STATUS', 'ORDER_DATE', 'Ki-67_Percent', 'Ki-67_Grade',
       'NOTE_TEXT'],
      dtype='object')

In [86]:
drop_columns = ['ORDER_DATE', 'STAGE_NUM', 'MARGIN_STATUS']
merge = merge.drop(drop_columns, axis=1)

In [87]:
merge.head(1)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,Ki-67_Percent,Ki-67_Grade,NOTE_TEXT
0,4333041185,50,454994176,2021-11-03,SCREEN,L,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaN,NaN,INDEX-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_subtype_info** <span style="color:red">(Assign missing value manually)</span>

In [88]:
df_step2 = merge.copy()

In [89]:
# df_step2["EXAM_COMPLETED_DATE"] = df_step2["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
# df_step2["PATHOLOGY_DATE"] = df_step2["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

In [90]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", "Cancer", 'mo_cancer_subtype_info' + ".xlsx")
df_step2.to_excel(output_file, index=False)

### **3. Assign subtype** (Luminal, HER2, TNBC)
#### **subtype**
* **Luminal A**: ER+, HER2-, low Ki-67 (< 20%), low grade, slow growing
* **Luminal B**: ER+, HER2-, high Ki-67 (>= 20%), intermediate/high histologic grade
* **HER2**: HER2+       
* **TNBC**: ER-, PR-, HER2-

<span style="color: red">**Note**: </span> Ki-67 information available in procedure note
http://www.ncbi.nlm.nih.gov/books/NBK583808/

#### **HISTROLOGY_GRADE**
express how normal or abnormal the cancer cells and their growth patterns appear
* **GX**: not possible to assess
* **G1** (Well-differentiated carcinomas): have relatively normal-looking cells that do not appear to be growing rapidly and are arranged in small tubules for ductal cancer and cords for lobular cancer. These cancers tend to grow and spread slowly and to have a better prognosis (outlook)
* **G2** (Moderately differentiated carcinomas): have cells and growth patterns that look a little more abnormal 
* **G3** (Poorly differentiated carcinomas): lack normal features. They tend to grow and spread faster and to have a worse prognosis

https://www.cancer.org/cancer/diagnosis-staging/tests/biopsy-and-cytology-tests/understanding-your-pathology-report/breast-pathology/breast-cancer-pathology.html
#### **STAGE_NUM**
Once the T, N, and M categories, the tumor grade, and ER, PR, and HER2 status have been determined, this information is combined to give the cancer an overall stage. Stages are expressed in Roman numerals from stage I (the least advanced) to stage IV (the most advanced). Non-invasive cancer (carcinoma in situ) is listed as stage 0
* **Stage 0**: Non-invasive cancer (carcinoma in situ) 
* **Stage 1**: least advanced
* **Stage 2A**
* **Stage 2B**
* **Stage 3A**
* **Stage 3C**
* **Stage 4**

In [91]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", "Cancer", 'mo_cancer_subtype_info' + ".xlsx")
df = pd.read_excel(output_file)

#### <span style="color:darkcyan;">**Function**</span>

In [92]:
def assign_subtypes(df):
    """
    Assigns cancer subtypes ('TNBC', 'Luminal B', 'Luminal A', 'HER2') 
    based on the provided receptor and grade rules.
    """
    
    # Use a copy to avoid SettingWithCopyWarning during complex indexing
    df_copy = df.copy()

    # Define the conditions in order of priority (most specific first)
    
    # Rule (HER2): if 'HER2NEU' = 'P', then 'subtype' = 'HER2'
    # We check for 'P' (Positive)
    her2_cond = (df_copy['HER2NEU'] == 'P')
    df_copy.loc[her2_cond, 'subtype'] = 'HER2'
    
    # Rule (Triple Negative): if all 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR', 'HER2NEU' show 'N'
    tbnc_cond = (df_copy['ESTROGEN_RECEPTOR'] == 'N') & \
                (df_copy['PROGESTERONE_RECEPTOR'] == 'N') & \
                (df_copy['HER2NEU'] == 'N')
    df_copy.loc[tbnc_cond, 'subtype'] = 'TNBC'
    
    # Rule (Luminal A): if 'ESTROGEN_RECEPTOR' = 'P', 'HER2NEU' = 'N', 'HISTOLOGY_GRADE' = 'GX', 'G1', 'G2'
    # We use .isin() for multiple grade values.
    luminal_a_cond = (df_copy['ESTROGEN_RECEPTOR'] == 'P') & \
                     (df_copy['HER2NEU'] == 'N') & \
                     (df_copy['Ki-67_Percent'] < 20)
    df_copy.loc[luminal_a_cond, 'subtype'] = 'Luminal A'

    # Rule 2 (Luminal B): if 'ESTROGEN_RECEPTOR' = 'P', 'HER2NEU' = 'N', 'HISTOLOGY_GRADE' = 'G3'
    # This should be checked after Luminal A or using a more robust hierarchy, 
    luminal_b_cond = (df_copy['ESTROGEN_RECEPTOR'] == 'P') & \
                     (df_copy['HER2NEU'] == 'N') & \
                     (df_copy['Ki-67_Percent'] >= 20)
    df_copy.loc[luminal_b_cond, 'subtype'] = 'Luminal B'

    luminal_cond = (df_copy['ESTROGEN_RECEPTOR'] == 'P') & \
                     (df_copy['HER2NEU'] == 'N') & \
                     (df_copy['Ki-67_Percent'].isna())
    df_copy.loc[luminal_cond, 'subtype'] = 'Luminal'
    
    return df_copy

#### <span style="color:blue;">**Main**</span>

In [93]:
# Initialize the 'Subtype' column with a default value
df['subtype'] = np.nan

In [94]:
df_w_subtype = assign_subtypes(df)

In [95]:
df_w_subtype['subtype'].unique()

array([nan, 'Luminal', 'Luminal A'], dtype=object)

In [96]:
subtype_counts = df_w_subtype.loc[idx_index].groupby('subtype').size()
subtype_group = subtype_counts.reset_index(name='SubtypeCount')

In [97]:
subtype_group

,subtype,SubtypeCount
0,Luminal,2
1,Luminal A,2


In [98]:
df_w_subtype.drop(columns=['NOTE_TEXT'], inplace=True)

#### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_subtype**

In [99]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", "Cancer", 'mo_cancer_subtype' + ".xlsx")
df_w_subtype.to_excel(output_file, index=False)

# **Step 2. Merge MO cancer subtype with DICOM file location**
### output: <span style="color:DodgerBlue;">**mo_cancer_dicom**</span>

#### <span style="color:#FF6347;">**READ**</span> **mo_cancer_subtype**

In [100]:
file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", "Cancer", 'mo_cancer_subtype' + ".xlsx")
df = pd.read_excel(file_path)

In [ ]:
file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", "dicom_tag" + ".xlsx")
dicom = pd.read_excel(file_path)

#### <span style=color:blue>**Main**</span>

In [102]:
df.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,Ki-67_Percent,Ki-67_Grade,subtype
0,4333041185,50,454994176,2021-11-03,SCREEN,L,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaN,NaN,INDEX-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4333041185,50,454994176,2021-11-03,SCREEN,R,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaN,NaN,INDEX-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4333041185,51,451483856,2022-08-05,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,353483.0,2022-09-02,Malignant,INDEX,ID,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,NaN,NaN,Luminal
3,4333041185,51,451483856,2022-08-05,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,353483.0,2022-09-02,Malignant,INDEX,II,G2,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,NaN,NaN,Luminal
4,4333041185,51,451483856,2022-08-05,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,354429.0,2022-08-30,Benign,INDEX,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [103]:
dicom.head(5)

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,NaN,SECURE,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,Hologic R2 ImageChecker CAD SC,1,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,ML,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R ML,71100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,XCCL,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R XCCL,71100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,LM,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L LM C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


In [104]:
dicom.rename(columns={'PatientID': 'PATIENT_STUDY_ID', 'AccessionNumber': 'ACCESSION_NUMBER', 'Side':'SIDE'}, inplace=True)
dicom.columns

Index(['PATIENT_STUDY_ID', 'PatientBirthDate', 'PatientAge',
       'ACCESSION_NUMBER', 'StudyDate', 'Study', 'SIDE', 'Series', 'View',
       'StudyDescription', 'SeriesDescription', 'SeriesNumber', 'FolderPath'],
      dtype='object')

In [105]:
common_cols = df.columns.intersection(dicom.columns).tolist()
common_cols

['PATIENT_STUDY_ID',
 'PatientAge',
 'ACCESSION_NUMBER',
 'StudyDate',
 'Study',
 'SIDE',
 'Series']

In [106]:
df_dicom = pd.merge(
    df,
    dicom,
    on = ['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'SIDE'],
    how='left'
)

In [107]:
df_dicom.drop(columns="Series_x", inplace=True)
df_dicom.rename(columns={'Series_y': 'Series'}, inplace=True)

In [108]:
df_dicom.columns

Index(['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate',
       'Study', 'SIDE', 'COMPOSITION_NAME', 'FINDING_LOCATION',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE', 'BX_ID',
       'PATHOLOGY_DATE', 'LESION_CLASS', 'Index', 'PATHOLOGY_CD',
       'HISTROLOGY_GRADE', 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR',
       'HER2NEU', 'STAGE_T', 'STAGE_N', 'STAGE_M', 'Ki-67_Percent',
       'Ki-67_Grade', 'subtype', 'PatientBirthDate', 'Series', 'View',
       'StudyDescription', 'SeriesDescription', 'SeriesNumber', 'FolderPath'],
      dtype='object')

In [109]:
df_step2 = df_dicom[['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'PatientBirthDate', 'StudyDate',
       'Study', 'SIDE', 
       'Series', 'View',
       'StudyDescription', 'SeriesDescription', 'SeriesNumber',
       'COMPOSITION_NAME', 'FINDING_LOCATION',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE', 
       'BX_ID',
       'PATHOLOGY_DATE', 'LESION_CLASS', 'Index', 'PATHOLOGY_CD',
       'HISTROLOGY_GRADE', 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR',
       'HER2NEU', 'STAGE_T', 'STAGE_N', 'STAGE_M', 'Ki-67_Percent',
       'Ki-67_Grade', 'subtype', 
       'FolderPath']]

#### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_dicom**

In [110]:
file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/Cancer/mo_cancer_dicom.xlsx"
df_step2.to_excel(file_path, index=False)

In [111]:
df_step2.shape

(168, 33)

# **Step 3. Merge MO cancer patients with EHR**
### output: <span style="color:DodgerBlue;">**mo_cancer_ehr** & **mo_cancer_ehr_@index_1**</span>

In [112]:
file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/Cancer/mo_cancer.xlsx"
df = pd.read_excel(file_path)

In [115]:
df.columns

Index(['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate',
       'Study', 'SIDE', 'Series', 'COMPOSITION_NAME', 'FINDING_LOCATION',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE', 'BX_ID',
       'PATHOLOGY_DATE', 'LESION_CLASS', 'Index'],
      dtype='object')

In [123]:
extract_cols = ['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate', 'EXAM_COMPLETED_DATE', 'Study', 'Index']

In [ ]:
df_patient_visit = df.drop_duplicates(
    subset=extract_cols, keep = 'first').reset_index(drop = True)[extract_cols]

In [161]:
df_patient_visit.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,EXAM_COMPLETED_DATE,Study,Index
0,4333041185,50,454994176,2021-11-03,2021-11-03,SCREEN,INDEX-1
1,4333041185,51,451483856,2022-08-05,2022-08-05,DIAG,INDEX
2,4333326250,47,70522003,2017-07-27,2017-07-27,SCREEN,INDEX-1
3,4333326250,47,77514004,2018-05-16,2018-05-16,DIAG,INDEX
4,4333957642,58,73086517,2017-07-11,2017-07-11,SCREEN,INDEX-1


#### <span style="color:#FF6347;">**READ**</span> **EHR** (hormonal_mens, risk_factors, patient_demo, vitals )

In [113]:
hormonal_mens = pd.read_excel(os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/", "Cancer", "Cleaned", "hormonal_mens" + ".xlsx"))
risk_factors = pd.read_excel(os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/", "Cancer", "Cleaned", "risk_factors" + ".xlsx"))
patient_demo = pd.read_excel(os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/", "Cancer", "Cleaned", "patient_demo" + ".xlsx"))
vitals = pd.read_excel(os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/", "Cancer", "Cleaned", "vitals" + ".xlsx"))

### **1. Merge BMI from vitals**

In [132]:
vitals.head(5)

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330018595,2019-06-24,2208.0,OZ,63.5,IN,24.06
1,4330018595,2020-03-03,2176.0,OZ,63.5,IN,23.71
2,4330018595,2020-06-23,2000.0,OZ,63.5,IN,21.80
3,4330018595,2020-07-07,2000.0,OZ,63.5,IN,21.80
4,4330018595,2020-08-11,2000.0,OZ,63.5,IN,21.80


In [162]:
df_patient_visit['EXAM_COMPLETED_DATE'] = pd.to_datetime(df_patient_visit['EXAM_COMPLETED_DATE'])
vitals['DATE_TAKEN']                    = pd.to_datetime(vitals['DATE_TAKEN'])

visit_sorted  = df_patient_visit.sort_values('EXAM_COMPLETED_DATE').reset_index(drop=True)
vitals_sorted = vitals.sort_values('DATE_TAKEN').reset_index(drop=True)

df_patient_visit_vital = pd.merge_asof(
    visit_sorted,
    vitals_sorted,
    left_on  = 'EXAM_COMPLETED_DATE',
    right_on = 'DATE_TAKEN',
    by       = 'PATIENT_STUDY_ID',
    # tolerance=pd.Timedelta(days=365),
    direction= 'nearest'
)

print(f"df_patient_visit : {len(df_patient_visit)} rows")
print(f"df_patient_visit_vital : {len(merged)} rows")

df_patient_visit : 16 rows
df_patient_visit_vital : 16 rows


In [163]:
df_patient_visit_vital.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER'], ignore_index=True, inplace=True)

In [164]:
df_patient_visit_vital.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4333041185,50,454994176,2021-11-03,2021-11-03,SCREEN,INDEX-1,2021-11-01,2483.2,OZ,67.0,IN,24.31
1,4333041185,51,451483856,2022-08-05,2022-08-05,DIAG,INDEX,2022-08-02,2392.0,OZ,66.0,IN,24.13
2,4333326250,47,70522003,2017-07-27,2017-07-27,SCREEN,INDEX-1,2017-07-27,4012.8,OZ,62.0,IN,45.87
3,4333326250,47,77514004,2018-05-16,2018-05-16,DIAG,INDEX,2018-05-02,3587.2,OZ,63.0,IN,39.72
4,4333957642,58,73086517,2017-07-11,2017-07-11,SCREEN,INDEX-1,2017-09-12,3206.4,OZ,65.5,IN,32.84


### **2. Merge hormonal_mens, risk_factors, patient_demo**

In [165]:
merge_hr = hormonal_mens.merge(risk_factors, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE'], how='inner')
merge_hrp = merge_hr.merge(patient_demo, on=['PATIENT_STUDY_ID'], how='inner')

### **3. Merge MO cancer patients with EHR**

In [166]:
merge_hrp['EXAM_COMPLETED_DATE'] = pd.to_datetime(merge_hrp['EXAM_COMPLETED_DATE'])

In [167]:
merge_hrp.head(1)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN
0,4330018595,63027507,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-07-26,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


In [168]:
df_patient_visit_ehr = pd.merge(df_patient_visit_vital, merge_hrp, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE'], how='left')

In [169]:
df_patient_visit_ehr

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN
0,4333041185,50,454994176,2021-11-03,2021-11-03,SCREEN,INDEX-1,2021-11-01,2483.2,OZ,67.00,IN,24.31,16,41,0,0,0,0,1.0,4.0,NaN,MENO,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1971-07-01,FEMALE,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,4333041185,51,451483856,2022-08-05,2022-08-05,DIAG,INDEX,2022-08-02,2392.0,OZ,66.00,IN,24.13,16,41,0,0,0,0,1.0,3.0,2022-03-01,MENO,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1971-07-01,FEMALE,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,4333326250,47,70522003,2017-07-27,2017-07-27,SCREEN,INDEX-1,2017-07-27,4012.8,OZ,62.00,IN,45.87,13,23,0,0,0,0,3.0,3.0,2018-05-15,NaN,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1970-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,4333326250,47,77514004,2018-05-16,2018-05-16,DIAG,INDEX,2018-05-02,3587.2,OZ,63.00,IN,39.72,13,23,0,0,0,0,3.0,3.0,2018-05-15,NaN,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1970-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
4,4333957642,58,73086517,2017-07-11,2017-07-11,SCREEN,INDEX-1,2017-09-12,3206.4,OZ,65.50,IN,32.84,14,0,48,48,0,0,0.0,0.0,2007-01-01,POSTSUR,0,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1959-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
5,4333957642,59,78139726,2018-07-12,2018-07-12,SCREEN,INDEX,2018-08-10,3312.0,OZ,66.75,IN,32.66,14,0,48,48,0,0,0.0,0.0,2007-01-01,POSTSUR,0,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1959-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
6,4333957642,59,77754683,2018-08-03,2018-08-03,DIAG,INDEX,2018-08-10,3312.0,OZ,66.75,IN,32.66,14,0,48,48,0,0,0.0,0.0,2007-01-01,POSTSUR,0,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1959-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
7,4334046824,77,78054829,2018-05-07,2018-05-07,SCREEN,INDEX-1,2019-06-05,1913.6,OZ,58.00,IN,25.00,13,0,40,40,40,40,0.0,0.0,NaN,POSTSUR,0,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1940-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
8,4334046824,78,63515140,2019-05-24,2019-05-24,DIAG,INDEX,2019-06-05,1913.6,OZ,58.00,IN,25.00,13,0,40,40,40,40,0.0,0.0,NaN,POSTSUR,0,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1940-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
9,4334064465,71,65500704,2018-11-

#### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_ehr, mo_cancer_ehr_@_index_1**

In [173]:
df_patient_visit_ehr["DATE_TAKEN"] = df_patient_visit_ehr["DATE_TAKEN"].dt.strftime("%Y-%m-%d")
df_patient_visit_ehr["EXAM_COMPLETED_DATE"] = df_patient_visit_ehr["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")

In [170]:
df_patient_visit_ehr['PATIENT_STUDY_ID'].nunique()

7

In [174]:
df_patient_ehr = df_patient_visit_ehr[df_patient_visit_ehr["Index"]=="INDEX-1"]

In [176]:
df_patient_ehr['PATIENT_STUDY_ID'].nunique()

7

In [179]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/", "Cancer", 'mo_cancer_ehr' + ".xlsx")
df_patient_visit_ehr.to_excel(output_file, index=False)

output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/", "Cancer", 'mo_cancer_ehr_@_index_1' + ".xlsx")
df_patient_ehr.to_excel(output_file, index=False)